# Hall effect 

In [6]:
#======= librerie =============#

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
%matplotlib inline

print("\nLibrerie caricate correttamente\n")


Librerie caricate correttamente



In [17]:
#======== definizioni ==========#

# =========================================================
# 1. FUNZIONI DI CARICAMENTO (Gestiscono virgole o punti e virgola)
# =========================================================
def load_csv_data(filepath):

    df = pd.read_csv(filepath, sep= ";", decimal=',', header = None)
    
    # Assumiamo che la colonna 0 sia B (o T) e la colonna 1 sia rho o RH
    x = df.iloc[:, 0].values.astype(float)
    y = df.iloc[:, 1].values.astype(float)
    
    # Ordiniamo in base a B crescente (fondamentale per le interpolazioni)
    sort_idx = np.argsort(x)
    return x[sort_idx], y[sort_idx]

# =========================================================
# 2. FUNZIONI FISICHE (Conversioni e Calcoli)
# =========================================================
def get_MR(rho_array):
    """Calcola la Magnetoresistenza MR = (rho(B) - rho(0)) / rho(0)"""
    rho_0 = rho_array[0] # Prende il valore a campo più basso (idealmente B=0)
    return (rho_array - rho_0) / rho_0

def get_rho_xy(RH_array, B_array, factor=0.1):
    """
    Calcola rho_xy = R_H * B.
    Se R_H è in mm^3/C e B in Tesla, il fattore 0.1 restituisce muOhm*cm.
    """
    return RH_array * B_array * factor

# =========================================================
# 3. LA FUNZIONE "MASTER" (Estrapola e Allinea tutto)
# =========================================================
def process_hall_dataset(file_rho, file_RH):
    """
    Prende in pasto i percorsi dei file rho(B) e RH(B) per una data T.
    Allinea R_H sull'asse B di rho(B), e restituisce tutte le grandezze pronte per i fit.
    """
    # 1. Caricamento dati grezzi
    df_RH = pd.read_csv(file_RH, sep=";", decimal=',', header=None)
    df_RH_clean = df_RH.groupby(0).mean().reset_index()
    
    B_RH = df_RH_clean.iloc[:, 0].values.astype(float)
    RH_raw = df_RH_clean.iloc[:, 1].values.astype(float)
    
    df_rho = pd.read_csv(file_rho, sep=";", decimal=',', header=None)
    df_rho_clean = df_rho.groupby(0).mean().reset_index()
    
    B_rho = df_rho_clean.iloc[:, 0].values.astype(float)
    rho_raw = df_rho_clean.iloc[:, 1].values.astype(float)
    
    if B_rho is None or B_RH is None:
        return None
    
    # 2. Interpolazione di R_H sui punti di campo magnetico B_rho
    # Usiamo interpolazione cubica per curve morbide. 'extrapolate' salva i bordi.
    RH_interp_func = interp1d(B_RH, RH_raw, kind='cubic', fill_value="extrapolate")
    RH_aligned = RH_interp_func(B_rho)
    
    # 3. Calcolo Grandezze Fisiche
    MR = get_MR(rho_xx)
    rho_xy = get_rho_xy(RH_aligned, B_rho, factor=0.1)
    
    # 4. Calcolo Cotangente dell'Angolo di Hall (cot(theta_H) = rho_xx / rho_xy)
    # Sopprimiamo i warning per la divisione per zero quando B=0 (rho_xy = 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        cot_theta_H = rho_xx / rho_xy
        # A B=0 l'angolo di Hall non esiste, mettiamo NaN per i grafici puliti
        cot_theta_H[B_rho == 0] = np.nan 
        
    # Restituiamo un dizionario o volendo un DataFrame Pandas comodissimo
    results = pd.DataFrame({
        'B_Tesla': B_rho,
        'rho_xx': rho_xx,          # muOhm cm
        'MR_adim': MR,             # adimensionale
        'RH_aligned': RH_aligned,  # mm^3 / C
        'rho_xy': rho_xy,          # muOhm cm
        'cot_theta_H': cot_theta_H # adimensionale
    })
    
    return results
print("\nDefinizioni registrate correttamente\n")


Definizioni registrate correttamente



In [18]:
# Imposta i percorsi corretti dei file (adatta il percorso se necessario)
file_rho_10K = "Data_sets/Nd-LSCO_p21/rho_Nd-LSCO_p21_10K.csv"
file_RH_10K  = "Data_sets/Nd-LSCO_p21/RH(B)_Nd-LSCO_p21_10K.csv"

# Magia: la funzione master fa tutto il lavoro sporco
df_10K = process_hall_dataset(file_rho_10K, file_RH_10K)

# Ora hai un Dataframe completo! Puoi plottare la cotangente di Hall:
import matplotlib.pyplot as plt

plt.plot(df_10K['B_Tesla'], df_10K['cot_theta_H'], 'o-', label="10 K")
plt.xlabel("Magnetic Field B (T)")
plt.ylabel(r"$\cot(\theta_H)$")
plt.legend()
plt.show()

NameError: name 'rho_xx' is not defined